# Analysis of native scan results

This tutorial analyzes genuine, bundled McStas scan outputs with `load_scan()` and `ScanResult.plot_overview()`. It does not build an instrument, call `mcrun`, or use any execution API. The plotting examples consume scalar monitor summaries only, so detector arrays remain unloaded.

The fixture outputs are small native captures. They include numeric, categorical, repeated, two-dimensional, and intentionally partial observations.

In [ ]:
import matplotlib
matplotlib.use("Agg")

from pathlib import Path
import matplotlib.pyplot as plt
import mcstasscript as ms

fixture_root = Path(ms.__file__).parent / "tests" / "scan_fixtures"
fixture_names = sorted(
    path.name for path in fixture_root.iterdir()
    if path.is_dir() and path.name != "__pycache__"
)
fixture_names

## Load observed data

Load the fixture directories after inspecting their names. The loader observes the points that exist; it does not reconstruct a requested plan from coordinates. `ascii_partial` deliberately contains a missing point result.

In [ ]:
numeric = ms.load_scan(fixture_root / "ascii_1d")
grid = ms.load_scan(fixture_root / "ascii_cartesian_2x3")
categorical = ms.load_scan(fixture_root / "ascii_categorical_repeated")
partial = ms.load_scan(fixture_root / "ascii_partial")
for name, scan in (("numeric", numeric), ("grid", grid),
                    ("categorical", categorical), ("partial", partial)):
    print(name, len(scan.points), scan.parameter_names, scan.monitors)

print("partial total_I:", partial.dataset("spectrum").total_I)

## Numeric one-dimensional observations

With one varying numeric parameter, `kind="auto"` selects a line in the observed point order. Error bars are taken from `total_E`; this is summary plotting and does not access `point.data`.

In [ ]:
figure, axes = numeric.plot_overview(
    monitor="spectrum", x="theta", show=False, marker="o")
axes.set_title("Numeric scan with summary error bars")
figure.tight_layout()
figure

## Categorical and repeated observations

Categorical coordinates use a scatter plot. Repeated coordinates are retained as separate observations rather than deduplicated or averaged. The exact slice below keeps the two repeated `category="vanadium"` observations.

In [ ]:
print([point.parameters for point in categorical.points])
figure, axes = categorical.plot_overview(
    monitor="spectrum", x="category", where={"theta": 0.0},
    show=False, s=80)
axes.set_title("Categorical observations, including repetition")
figure.tight_layout()
figure

## Partial observations

A missing scalar total is not silently changed to zero. The missing intensity remains absent in the overview, and an unavailable error remains distinct from an observed zero error.

In [ ]:
partial_view = partial.dataset("spectrum")
print("total_I:", partial_view.total_I)
print("total_E:", partial_view.total_E)
figure, axes = partial.plot_overview(
    monitor="spectrum", x="theta", show=False, marker="o")
axes.set_title("Partial scan: missing summary values stay missing")
figure.tight_layout()
figure

## Two-dimensional views and exact slices

Two explicit coordinates default to a colored scatter plot. A heatmap is opt-in and validates the observed numeric coordinate pairs; it does not invent cells for a requested grid. `where` selects exact stored parameter values for a one-dimensional slice.

In [ ]:
figure, (scatter_axes, heatmap_axes, slice_axes) = plt.subplots(1, 3, figsize=(14, 4))
grid.plot_overview(
    monitor="spectrum", x="theta", y="wavelength",
    ax=scatter_axes, show=False)
grid.plot_overview(
    monitor="spectrum", x="theta", y="wavelength",
    kind="heatmap", ax=heatmap_axes, show=False, cmap="viridis")
scatter_axes.set_title("Observed pairs")
heatmap_axes.set_title("Explicit heatmap")
figure.tight_layout()

grid.plot_overview(
    monitor="spectrum", x="theta",
    where={"wavelength": 4.0}, ax=slice_axes,
    show=False, marker="o")
slice_axes.set_title("Exact wavelength slice")
figure.tight_layout()
figure